# M3b Tier 2 training: one model per session (PLAN §7 M3b; D-009, D-072)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**
(and **GITHUB_TOKEN** only if the repo is private). Run it as **Save Version → Save & Run All**
so it keeps going without the browser (≤ 12 h); read the output from the version's log.

Set `MODEL` in the first code cell: session 1 `qwen2.5-0.5b` (full s0, LoRA s0, full s1, s2),
session 2 `llama3.2-1b` (full s0, LoRA s0). The notebook:
1. restores this model's finished jobs from the private store (a re-run skips them);
2. runs an 8-minute full-FT probe and stops unless the session bound is ≤ 11 h (D-072);
3. trains every pending u||h pair, syncing each finished pair to the private store;
4. runs DC-07 on the real endpoints, syncs, and prints an aggregate summary (no text).

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
MODEL = "qwen2.5-0.5b"  # session 1: qwen2.5-0.5b; session 2: llama3.2-1b
CFG = f"configs/tier2_{MODEL}.yaml"
PROBE = f"configs/m3b_probe_{MODEL}.yaml"

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = dict(os.environ)
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_m3b.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

print(load_kaggle_secrets(["HF_TOKEN"]))
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore this model's finished jobs (done.json, finals) from the private store, then select
# the data (ftdata ids are deterministic, so train keys match the earlier session).
import json
from rbbd import config as _config
_cfg = _config.load(CFG)
SLUG = _cfg["models"][0]["slug"]
MODEL_ID = _cfg["models"][0]["id"]
!python -m rbbd.cli restore --path train/{SLUG} || echo "nothing to restore yet"
!python -m rbbd.cli run --config {CFG} --stages ftdata
!python -c "from rbbd.models.loading import LoadSpec, download; print(download(LoadSpec('{MODEL_ID}')))"

In [ ]:
# Session probe (D-072): 8 min of full FT, u||h. Bound = harmful 3-epoch hours x pending pairs.
import glob, subprocess, time
from pathlib import Path
from rbbd import runner
from rbbd.finetune import sft
from rbbd.utils import manifest as mf
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "rbbd.cli", "run", "--config", PROBE, "--stages", "ftdata,train"],
                   capture_output=True, text=True)
print(f"probe exit {r.returncode} after {time.time() - t0:.0f} s"); print(r.stdout[-800:])
if r.returncode:
    print(r.stderr[-3000:])
probe_slug = _config.load(PROBE)["models"][0]["slug"]
PROBE_DONE = {}
for path in glob.glob(f"/kaggle/working/artifacts/train/{probe_slug}/full/*/seed0/*/done.json"):
    PROBE_DONE[path.split("/")[-4]] = json.load(open(path))
root = "/kaggle/working/artifacts"
ft = mf.read(mf.manifest_path(Path(root), "ftdata", runner.stage_run_keys(_cfg)["ftdata"]))
jobs = sft.plan_jobs(_cfg, Path(root), list(ft.output_hashes))
pending_pairs = len([j for j in jobs if j.split == "harmful" and not (j.out_dir / "done.json").exists()])
h3 = (PROBE_DONE.get("harmful") or {}).get("projected_hours", {}).get("3_epochs")
BOUND = None if h3 is None else round(h3 * pending_pairs, 2)
PROCEED = BOUND is not None and BOUND <= 11.0
print(json.dumps({"probe": {k: {f: v.get(f) for f in ("tokens_per_second_steady", "peak_mem_gib",
      "projected_hours", "global_step")} for k, v in PROBE_DONE.items()},
      "pending_pairs": pending_pairs, "bound_hours": BOUND, "proceed": PROCEED}, indent=1))

In [ ]:
# The long run: every pending u||h pair, each finished pair synced to the store (D-072).
if PROCEED:
    !python -m rbbd.cli run --config {CFG} --stages ftdata,train
else:
    print("NOT RUNNING: session bound > 11 h or probe failed; paste the probe cell output (D-072)")

In [ ]:
# DC-07 on the real Tier 2 endpoints (full: exact α ∈ {0, 1}; LoRA: adapters + linearity).
if PROCEED:
    os.environ["RBBD_M3B_CONFIG"] = CFG
    !pytest -q -m gpu -rfE --tb=short tests/gpu/test_train_gpu.py -k "real_full or tier2_lora" 2>&1 | tee $ENV_DIR/pytest_gpu_m3b.txt | tail -30

In [ ]:
# End-of-session sync (private store, D-041): this model's jobs, ids, manifests, env JSON.
for _path in ["manifests", "ftdata", f"train/{SLUG}", f"env/{RBBD_SESSION_ID}"]:
    if os.path.isdir(f"/kaggle/working/artifacts/{_path}"):
        !python -m rbbd.cli sync --path {_path}

In [ ]:
# Summary to paste back (aggregates only).
print("bound_hours", BOUND, "proceed", PROCEED)
for path in sorted(glob.glob(f"/kaggle/working/artifacts/train/{SLUG}/*/*/*/*/done.json")):
    d = json.load(open(path))
    print(path.split("/train/")[1]); print(json.dumps({k: d.get(k) for k in ("global_step", "epochs",
          "train_loss", "final_loss", "tokens_per_second_steady", "peak_mem_gib", "scaler_skipped_steps",
          "resumed_from", "seconds_this_process")}))
for path in sorted(glob.glob(f"/kaggle/working/artifacts/train/{SLUG}/*/*/*/*/oom.json")):
    print("OOM:", path.split("/train/")[1], open(path).read()[:200])
for path in sorted(glob.glob(f"{ENV_DIR}/m3a_train_gpu.json")):
    print(open(path).read())
!df -h /kaggle/working | tail -1
!python -m rbbd.cli status --config {CFG}